In [ ]:
from os.path import join, exists
from os import mkdir
from torch.nn import functional as F
from torchvision import transforms
import torch as th
from PIL import Image
from tqdm import tqdm

In [ ]:
nb_bytes = (6330 * 8181 + 9506 * 14830 + 5249 * 7606) * (1 + 2 * 64)
print(nb_bytes / 2**30)

In [ ]:
zip_path = "/home/samuel/Téléchargements/vesuvius-challenge-ink-detection.zip"

In [ ]:
output_dir = "/media/samuel/M2_nvme_Sam/vesuvius-challenge-ink-detection"

In [ ]:
if not exists(output_dir):
    mkdir(output_dir)

In [ ]:
if exists(zip_path) and not exists(join(output_dir, "train")):
    !unzip $zip_path -d $output_dir

In [ ]:
DESIRED_SIZE = (256, 256)

to_tensor = transforms.ToTensor()

extracted_tensor_path = join(output_dir, "train_tensors")
if not exists(extracted_tensor_path):
    mkdir(extracted_tensor_path)

idx = 0

for img_idx in range(1, 4):
    img_folder = join(output_dir, "train", str(img_idx))
    
    print("=============")
    print(f"Image idx {img_idx}")
    print()
    
    # read label
    label = join(img_folder, "inklabels.png")
    label_t = (
        F.unfold(
            to_tensor(Image.open(label))[None],
            DESIRED_SIZE, 1, 0, DESIRED_SIZE
        )
        .view(1, DESIRED_SIZE[0], DESIRED_SIZE[1], -1)
        .permute(3, 0, 1, 2)
        .gt(0)
    )
    
    print("label split : OK")
    
    # read mask : don't save full empty sub-image
    mask = join(img_folder, "mask.png")
    mask_t = (
        F.unfold(
            to_tensor(Image.open(mask))[None],
            DESIRED_SIZE, 1, 0, DESIRED_SIZE
        )
        .view(DESIRED_SIZE[0] * DESIRED_SIZE[1], -1)
        .permute(1, 0)
        .gt(0)
        .any(dim=1)
    )
    
    print("mask split : OK")
    print("saving label...")
    
    label_idx = idx
    for i in tqdm(range(label_t.size(0))):
        if bool(mask_t[i]):
            th.save(
                label_t[i].clone(), join(extracted_tensor_path, f"lbl_{label_idx}.pt")
            )
    
            label_idx += 1
    
    slices_t = []
    
    print("reading slices...")
    
    for slice_idx in tqdm(range(1, 65)):
        slice_path = join(img_folder, "surface_volume", f"{slice_idx:02}.tif")
        
        slices_t.append(
            F.unfold(
                to_tensor(Image.open(slice_path)).to(th.float32),
                DESIRED_SIZE, 1, 0, DESIRED_SIZE
            )
            .to(th.int16)
            .view(1, DESIRED_SIZE[0], DESIRED_SIZE[1], -1)
            .permute(3, 0, 1, 2)
        )
    
    print("saving slices...")
    
    img_idx = idx
    for i in tqdm(range(slices_t[0].size(0))):
        if bool(mask_t[i]):
            curr_slice = th.stack(
                [s[i] for s in slices_t],
                dim=-1,
            )
            th.save(
                curr_slice.clone(), join(extracted_tensor_path, f"img_{img_idx}.pt")
            )
            
            img_idx += 1
    
    assert label_idx == img_idx
    
    idx = img_idx
    print(f"patchs done : {idx}")